# Nexus sleeve attribution (v2)

Event-walk gross (matches dashboard within ~$300) is split into six sleeves:

`overnight + timing + pca_held_mr + fill_mr + mark_to_fill + unexecuted`

Unexecuted includes market return, Base2 outbound entry BV, and EOD align.
`validation_error = ours − event_walk_gross` must be ~0 (never plugged).


In [1]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

ROOT = Path('/home/ajain/apeksha/pnl_attribution/src/v2')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import CONFIG, get_ch_client
from scripts.library import load_library_eod_pnl
from scripts.position_timeline import load_bod_spreads, reconcile_timeline_vs_library
from scripts.sleeve_attribution import (
    SLEEVE_COLS,
    assign_transfers_to_cycles,
    attribute_day_sleeves,
    attribute_range_vs_library,
    attribute_range_by_cycle,
    compute_timing_mr,
)

client = get_ch_client()
BOOK = 'Hedger Spreads'
DAY = date(2026, 10, 6)           # single-day cells
RANGE_START = date(2026, 9, 17)   # per-day component table
RANGE_END = DAY                  # or date.today()

print('books =', len(CONFIG.book_names))
print('book =', BOOK, '| day =', DAY)
print('range', RANGE_START, '→', RANGE_END)


books = 10
book = Hedger Spreads | day = 2026-10-06
range 2026-09-17 → 2026-10-06


### Inventory + library recon

BOD spreads and event-walk gross vs library (`overnight + m2m + trade`).


In [2]:
bod_ts, bod = load_bod_spreads(client, DAY, BOOK)
print('BOD curve', bod_ts)
display(bod.to_frame('lots').T)

recon = reconcile_timeline_vs_library(client, DAY, BOOK)
display(recon)
lib = load_library_eod_pnl(client, DAY, BOOK)
print('library', lib)


BOD curve 2026-10-06 05:21:53.211000


,2026-12-01,2027-01-01,2027-02-01,2027-03-01,2027-04-01,2027-05-01,2027-06-01,2027-07-01,2027-08-01,2027-09-01,...,2028-08-01,2028-09-01,2028-10-01,2028-11-01,2028-12-01,2029-01-01,2029-02-01,2029-03-01,2029-04-01,2029-05-01
lots,9.0,5.0,-4.0,0.0,0.0,-1.0,-36.0,32.0,0.0,1.0,...,-16.0,-44.0,0.0,-45.0,0.0,1.0,0.0,-1.0,-1.0,-1.0


,ours,library,diff
component,,,
overnight,193.21,883.318783,-690.11
m2m_pnl,-22768.30,-23462.620666,694.32
overnight+m2m,-22575.09,-22579.300000,4.21
trade_pnl,-22366.79,-22366.787647,-0.00
gross,-44941.88,-44946.089531,4.21


library {'overnight': 883.3187829746693, 'm2m_pnl': -23462.62066629612, 'trade_pnl': -22366.78764732335, 'gross': -44946.0895306448}


### Assign transfers + timing MR

- Keep recorded `cycle_id`
- Untagged → first intention after the transfer
- Never assign Hedger → Spreadgr Base 2 rollers
- `timing_mr = lots × 1000 × (curve_cycle − curve_xfer)`


In [3]:
assigned = assign_transfers_to_cycles(client, BOOK, RANGE_START, RANGE_END)
assigned = compute_timing_mr(client, assigned)
print(assigned['assigned_from'].value_counts(dropna=False).to_dict())
print('timing_mr sum', round(float(assigned['timing_mr'].sum()), 2))
display(assigned.head(10))


{'next_intention': 9326, 'recorded': 1283, 'hedger_to_base2': 214}
timing_mr sum -7521.74


,timestamp,id,desk,source_book,trader,side,quantity,start_tenor,end_tenor,price,cycle_id_recorded,outbound,cycle_id_assigned,cycle_ts,assigned_from,lag_ms,timing_mr
0,2026-09-17 06:12:39.423,37673355,ProNap,Hedger Spreads,Edward Himbury,buy,43,2027-01-01,2027-02-01,3.20487,NaN,False,e61b65dc-d67c-4b85-9076-5ec383a55a67,2026-09-17 06:12:41.508,next_intention,2085.0,639.27
1,2026-09-17 06:14:09.280,37673469,ProNap,Hedger Spreads,Edward Himbury,buy,21,2027-01-01,2027-02-01,3.18524,NaN,False,52e4df80-44da-4b3f-ac9e-201104366b48,2026-09-17 06:14:09.567,next_intention,287.0,0.00
2,2026-09-17 06:33:02.513,37673826,Fuel,Hedger Spreads,Jamie Challis,buy,15,2026-11-01,2026-12-01,4.48065,NaN,False,76b11f0d-d245-4875-853d-655653522efb,2026-09-17 06:33:02.924,next_intention,411.0,0.00
3,2026-09-17 06:33:02.517,37673829,Fuel,Hedger Spreads,Jamie Challis,buy,1,2026-11-01,2027-01-01,8.10945,NaN,False,76b11f0d-d245-4875-853d-655653522efb,2026-09-17 06:33:02.924,next_intention,407.0,0.00
4,2026-09-17 06:33:59.732,37673906,GasNap,Hedger Spreads,Neema Shadpour,buy,5,2026-11-01,2026-12-01,4.37167,NaN,False,15de9f1f-f7fe-487d-a21c-0580187ebc3d,2026-09-17 06:34:00.024,next_intention,292.0,0.00
5,2026-09-17 06:34:46.578,37673970,Fuel,Hedger Spreads,Jamie Challis,sell,5,2026-11-01,2026-12-01,4.40000,NaN,False,c7f68155-6c28-4825-a6b0-d3bdf6aa65ea,2026-09-17 06:34:47.100,next_intention,522.0,0.00
6,2026-09-17 06:42:05.473,37674059,Fuel,Hedger Spreads,Jamie Challis,buy,5,2026-11-01,2026-12-01,4.53000,NaN,False,2e83bd99-3010-43e9-a7e8-b015e1a5688f,2026-09-17 06:42:05.692,next_intention,219.0,0.00
7,2026-09-17 06:47:33.898,37674124,Fuel,Hedger Spreads,Jamie Challis,buy,6,2026-12-01,2027-01-01,3.63991,NaN,False,2b911016-aa1f-4e3b-b3fd-81f97d927127,2026-09-17 06:47:34.234,next_intention,336.0,0.00
8,2026-09-17 06:47:47.890,37674146,Fuel,Hedger Spreads,Jamie Challis,buy,5,2026-11-01,2026-12-01,4.53000,NaN,False,89e18308-d276-487f-a572-235220977d67,2026-09-17 06:47:48.129,next_intention,239.0,0.00
9,2026-09-17 06:54:52.069,37674303,GasNap,Hedger Spreads,Neema Shadpour,buy,7,2026-11-01,2026-12-01,4.45909,NaN,False,9b596ef3-3fb9-4213-b719-63faea8111da,2026-09-17 06:54:52.588,next_intention,519.0,5.30


### Day sleeves from the event walk

Every recon dollar assigned once. Subcolumns shown for timing and unexecuted.


In [ ]:
tl_mr, totals = attribute_day_sleeves(client, DAY, BOOK, assigned)
keys = [
    'overnight', 'timing_mr', 'pca_held_mr', 'fill_mr', 'mark_to_fill',
    'unexecuted_mr',
    'ours', 'event_walk_gross', 'library_gross', 'validation_error',
]
for k in keys:
    print(f'{k:28s} {totals[k]:>14}')
print(f"{'ours_vs_lib':28s} {round(totals['ours'] - totals['library_gross'], 2):>14}")


### Per-day sleeve components

One row per library EOD day. `validation_error` should be ~0; `ours_vs_lib` is vs dashboard gross.


In [2]:
cmp = attribute_range_vs_library(client, BOOK, RANGE_START, end=RANGE_END)
cmp = cmp.copy()
cmp['ours_vs_lib'] = (cmp['ours'] - cmp['library_gross']).round(2)

cols = [
    'date',
    'overnight',
    'timing_mr',
    'pca_held_mr',
    'fill_mr',
    'mark_to_fill',
    'unexecuted_mr',
    'ours',
    'event_walk_gross',
    'library_gross',
    'validation_error',
    'ours_vs_lib',
]
view = cmp[[c for c in cols if c in cmp.columns]]
display(view.style.format(precision=2))

if not cmp.empty:
    print('days', len(cmp),
          '| max |validation_error|', round(cmp['validation_error'].abs().max(), 2),
          '| max |ours_vs_lib|', round(cmp['ours_vs_lib'].abs().max(), 2))

Hedger Spreads sleeves: 100%|██████████| 20/20 [19:09<00:00, 57.46s/day]


,date,overnight,timing_mr,pca_held_mr,fill_mr,mark_to_fill,unexecuted_mr,ours,event_walk_gross,library_gross,validation_error,ours_vs_lib
0,2026-09-17,669.71,-3862.68,16342.14,-48123.40,11544.26,-109725.24,-133155.21,-133155.20,-133085.27,-0.01,-69.94
1,2026-09-18,-1130.50,3864.72,14950.03,-12986.85,-1433.45,-23221.71,-19957.76,-19957.76,-20048.01,0.00,90.25
2,2026-09-21,87.24,10790.88,8688.89,-7513.72,-7933.08,-14464.32,-10344.11,-10344.12,-10344.13,0.01,0.02
3,2026-09-22,-1409.30,-6531.05,6852.32,-35538.11,5267.54,-4148.66,-35507.26,-35507.26,-35507.26,0.00,0.00
4,2026-09-23,-1586.51,741.04,20470.13,-22008.80,2623.54,-11213.50,-10974.10,-10974.10,-10974.09,0.00,-0.01
5,2026-09-24,-714.95,12694.35,-5813.68,337.42,-5247.22,3899.61,5155.53,5155.54,4842.79,-0.01,312.74
6,2026-09-25,19182.61,3156.42,-5447.00,-10277.43,-4691.52,-10685.67,-8762.59,-8762.59,-8762.59,0.00,0.00
7,2026-09-28,-8678.46,-3088.99,2692.54,-14232.27,-759.09,-30132.14,-54198.41,-54198.40,-54128.42,-0.01,-69.99
8,2026-09-29,-893.25,-5538.49,11128.97,-23360.79,-1443.35,-7202.37,-27309.28,-27309.28,-27380.31,0.00,71.03
9,2026-09-30,-2826.04,-8201.32,9316.99,-19030.06,316.93,-27888.61,-48312.11,-48312.11,-48314.99,0.00,2.88


days 14 | max |validation_error| 0.01 | max |ours_vs_lib| 312.74


In [5]:
view.set_index('date').sum()

overnight             1286.40
timing_mr             3172.75
pca_held_mr         145617.43
fill_mr            -246669.10
mark_to_fill         -3223.27
unexecuted_mr      -332986.52
ours               -432802.31
event_walk_gross   -432802.29
library_gross      -433295.92
validation_error        -0.02
ours_vs_lib            493.61
dtype: float64

### Per-cycle PnL (all days in range)

One row per intention cycle. Includes timing / PCA / fill_mr / mark_to_fill / unexecuted for that cycle only — not overnight, Base2 outbound BV, or inter-cycle gaps.


In [ ]:
# Per-cycle PnL across RANGE_START → RANGE_END
# Cycle-attributable sleeves only (no overnight / Base2 / inter-cycle gaps).
by_cycle = attribute_range_by_cycle(client, BOOK, RANGE_START, end=RANGE_END)
cols = [
    'date', 'cycle_id', 't0', 't1',
    'timing_mr', 'pca_held_mr', 'fill_mr', 'mark_to_fill',
    'unexecuted_mr', 'cycle_pnl',
]
display(by_cycle[[c for c in cols if c in by_cycle.columns]].style.format({
    c: '{:.2f}' for c in [
        'timing_mr', 'pca_held_mr', 'fill_mr', 'mark_to_fill',
        'unexecuted_mr', 'cycle_pnl',
    ] if c in by_cycle.columns
}))
print('cycles', len(by_cycle),
      '| days', by_cycle['date'].nunique() if len(by_cycle) else 0,
      '| sum cycle_pnl', round(float(by_cycle['cycle_pnl'].sum()), 2) if len(by_cycle) else 0)


Hedger Spreads cycles:   0%|          | 0/20 [00:00<?, ?day/s]